# Krok 6. Wydarzenia z potwierdzonym efektem (E2)
**Wynik z instrukcji:** mnożnik każdego wydarzenia (święta, długie weekendy, okazje) per grupa i długość okna (ile dni przed i po); kryterium: efekt w obu latach 2023-2024 (do ustalenia). Poziom: `AsId` × wydarzenie. Dla wariantów `baseline_1` (Wielkanoc) i `baseline_2`.

**Jak mierzymy efekt dnia.** Dla każdego dnia porównujemy sprzedaż z **normą dnia**: medianą tego samego dnia tygodnia z dni czystych w oknie ±6 tygodni. Iloraz „sprzedaż / norma” to mnożnik dnia (1,0 = zwyczajnie, 1,8 = o 80 % więcej).

**Wydarzenie i okno.** Każde wydarzenie ma datę (np. Wielkanoc, 8 marca) i patrzymy na dni od 9 dni przed do 3 dni po (przesunięcia `offset`). Dla wydarzenia z kilkoma terminami w roku (długie weekendy) bierzemy średnią z terminów.

**Kryterium potwierdzenia (do ustalenia, proponowane tu).** Efekt dla asortymentu, wydarzenia i przesunięcia jest **potwierdzony**, gdy w **obu** latach (2023 i 2024) mnożnik jest po tej samej stronie jedynki i odchyla się od niej bardziej, niż zwykle odchylają się dni zwyczajne tego asortymentu (próg to 90. percentyl |ln mnożnika| na dniach czystych). Dwa lata to jedyne zabezpieczenie przed przypadkiem, więc wynik jest ostrożny, ale nie dowodzi niczego na pewno.

**Wynik zapisany do:** `dane/eda/wynik_06_wydarzenia.csv` (tylko potwierdzone) i `dane/eda/wynik_06_okna.csv` (długość okna na asortyment i wydarzenie).

In [6]:
# Ustawienia wspólne: moduł eda_wspolne.py leży obok tego notebooka
import os, sys
sys.path.insert(0, os.getcwd())
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import eda_wspolne as ew

ew.wejdz_do_korzenia()      # przechodzi do katalogu głównego projektu (tam, gdzie jest plik z danymi)
ew.styl()
print("Katalog projektu:", os.getcwd(), "| EDA używa danych do", ew.KONIEC_EDA.date(), "(rok testowy", ew.ROK_TEST, "pomijamy)")

Katalog projektu: /Users/mixer/python/analiza_sprzedazy_w_sklepie | EDA używa danych do 2024-12-31 (rok testowy 2025 pomijamy)


## Norma dnia
#### Objaśnienie
**Po co.** Żeby ocenić efekt wydarzenia, potrzebujemy wartości odniesienia: ile asortyment sprzedałby w ten dzień „normalnie”. Bierzemy medianę tego samego dnia tygodnia (np. wszystkich wtorków) z dni czystych w oknie ±6 tygodni. Mediana jest odporna na pojedyncze skoki. Norma korzysta z dni po obu stronach daty, więc służy **tylko do opisu lat EDA**, nigdy jako cecha prognozy (to byłby wyciek przyszłości).
**Co robi kod.** Buduje `R` (dni × asortymenty): iloraz sprzedaży i normy dla dni otwartych Pn-So. Dla niedzieli handlowej normą jest norma sobotniego dnia poprzedniego, więc jej mnożnik oznacza „niedziela handlowa / zwykła sobota”.
**Próg szumu.** Dla każdego asortymentu liczymy `prog` = 90. percentyl |ln R| na dniach czystych, czyli jak mocno odchylają się dni, w których nic się nie dzieje.
**Jak czytać wynik.** Mediana R na dniach czystych powinna wynosić około 1,0. Duży `prog` (np. 0,4) oznacza asortyment szumny: efekt wydarzenia musi być duży, żeby został uznany.

In [7]:
UZYJ_PROMO = ew.ma_kolumne("CzyPromocja")
fa = ew.wczytaj_sprzedaz(("CzyPromocja",) if UZYJ_PROMO else ())
kal = ew.kalendarz()
if UZYJ_PROMO:
    fa = fa[fa["CzyPromocja"] != 1]
pan = ew.panel_dzienny(fa, kal, "AsId")
del fa
pan["SKLEP"] = pan.sum(axis=1)
pan.columns = pan.columns.astype(str)

norma = ew.norma_dnia(pan, kal)
otwarty = ~kal["Zamkniety"]
Rw = np.full(pan.shape, np.nan)
zwykly = otwarty & (kal["DzienTyg"] < 6)
dobre = zwykly.values[:, None] & (norma.values > 0)
Rw[dobre] = (pan.values / np.where(norma.values > 0, norma.values, np.nan))[dobre]
nd = (kal["NdHandlowa"] & otwarty & (kal["DzienTyg"] == 6)).values
sob = norma.shift(1).values
dn = nd[:, None] & (sob > 0)
Rw[dn] = (pan.values / np.where(sob > 0, sob, np.nan))[dn]
R = pd.DataFrame(Rw, index=pan.index, columns=pan.columns)

czysty = kal["Czysty"].values
lnR = np.log(np.clip(R.values, 0.05, None))
prog = np.nanquantile(np.abs(lnR[czysty]), 0.90, axis=0)
print("Mediana R na dniach czystych (sklep):", round(float(np.nanmedian(R.values[czysty, -1])), 3))
print("Próg szumu |ln R| (90. percentyl): sklep =", round(float(prog[-1]), 3), "| mediana po asortymentach =", round(float(np.median(prog[:-1])), 3))

Mediana R na dniach czystych (sklep): 1.0
Próg szumu |ln R| (90. percentyl): sklep = 0.103 | mediana po asortymentach = nan


/opt/anaconda3/envs/analiza_sprzedazy_env/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,


## Lista wydarzeń
#### Objaśnienie
**Po co.** Definiujemy, jakie wydarzenia sprawdzamy. To jest lista **do edycji**: dopisz własne okazje (np. lokalny odpust), a notebook policzy je tak samo.
**Rodzaje.** (1) święta ze stałą datą, (2) święta ruchome liczone od Wielkanocy (Wielkanoc, Boże Ciało, tłusty czwartek), (3) okazje handlowe ze stałą datą (Walentynki, Dzień Kobiety, Dzień Matki, Mikołajki i inne), (4) długie weekendy: każdy blok kolejnych dni z `CzyDlugiWeekend` to jedno wystąpienie, a datą jest jego pierwszy dzień.
**Jak czytać wynik.** Dla każdego wydarzenia widać daty w 2023 i 2024. Jeśli w którymś roku wydarzenie nie ma żadnej daty, nie da się go potwierdzić w obu latach.

In [8]:
from dateutil.easter import easter
STALE = {   # nazwa: (miesiąc, dzień)
    "Nowy Rok": (1, 1), "Trzech Króli": (1, 6), "Walentynki": (2, 14), "Dzień Kobiet": (3, 8), "Święto Pracy": (5, 1),
    "Konstytucja 3 maja": (5, 3), "Dzień Matki": (5, 26), "Dzień Dziecka": (6, 1), "Dzień Ojca": (6, 23),
    "Wniebowzięcie": (8, 15), "Halloween": (10, 31), "Wszystkich Świętych": (11, 1), "Zaduszki": (11, 2),
    "Święto Niepodległości": (11, 11), "Andrzejki": (11, 29), "Mikołajki": (12, 6), "Wigilia": (12, 24),
    "Boże Narodzenie": (12, 25), "Drugi dzień BN": (12, 26), "Sylwester": (12, 31),
}
RUCHOME = {"Wielkanoc": 0, "Boże Ciało": 60, "Tłusty czwartek": -52}     # dni względem Wielkanocy
LATA = (2023, 2024)

def terminy():
    ev = {}
    for nazwa, (m, d) in STALE.items():
        ev[nazwa] = {r: [pd.Timestamp(r, m, d)] for r in LATA}
    for nazwa, przes in RUCHOME.items():
        ev[nazwa] = {r: [pd.Timestamp(easter(r)) + pd.Timedelta(days=przes)] for r in LATA}
    dw = kal["DlugiWeekend"].values
    start = [kal.index[i] for i in range(len(dw)) if dw[i] and (i == 0 or not dw[i - 1])]
    ev["Długi weekend"] = {r: [d for d in start if d.year == r] for r in LATA}
    ev["Niedziela handlowa"] = {r: [d for d in kal.index[nd] if d.year == r] for r in LATA}
    return ev
WYDARZENIA = terminy()
print(pd.DataFrame({n: {r: len(v[r]) for r in LATA} for n, v in WYDARZENIA.items()}).T.rename(columns=lambda r: f"terminów {r}").to_string())

                       terminów 2023  terminów 2024
Nowy Rok                           1              1
Trzech Króli                       1              1
Walentynki                         1              1
Dzień Kobiet                       1              1
Święto Pracy                       1              1
Konstytucja 3 maja                 1              1
Dzień Matki                        1              1
Dzień Dziecka                      1              1
Dzień Ojca                         1              1
Wniebowzięcie                      1              1
Halloween                          1              1
Wszystkich Świętych                1              1
Zaduszki                           1              1
Święto Niepodległości              1              1
Andrzejki                          1              1
Mikołajki                          1              1
Wigilia                            1              1
Boże Narodzenie                    1              1
Drugi dzień 

## Mnożniki wydarzeń i potwierdzenie
#### Objaśnienie
**Po co.** Dla każdej trójki (wydarzenie, przesunięcie, asortyment) liczymy mnożnik w 2023 i 2024 i sprawdzamy kryterium potwierdzenia.
**Co robi kod.**
- `ratio[e, rok, o]` to wektor mnożników po asortymentach: średnia (`nanmean`) z terminów wydarzenia w danym roku, w dniu `termin + o`. Dni zamknięte dają `NaN`.
- Mnożnik wynikowy to średnia z dwóch lat, a **potwierdzenie** wymaga, żeby w obu latach |ln mnożnika| przekraczało `prog` asortymentu i miało ten sam znak.
- **Okno przed i po**: ile kolejnych dni z potwierdzonym efektem sięga od dnia wydarzenia wstecz (−1, −2, …) i w przód (+1, +2, …).
**Uwaga o nakładaniu się.** Wydarzenia leżą blisko siebie (np. Mikołajki, Wigilia, Sylwester), więc ich efekty się nakładają i nie da się ich w pełni rozdzielić. Okno z kilkoma wydarzeniami obok siebie interpretuj łącznie.
**Jak czytać wynik.** Dla sklepu drukujemy potwierdzone wydarzenia z oknem i największym mnożnikiem. Zwróć uwagę na duży mnożnik tylko w jednym roku: taki efekt nie przechodzi testu i nie trafia do wyniku.

**Przesunięcie** (offset) to liczba dni od daty wydarzenia. Mówi, który dzień wokół wydarzenia oceniamy.

W tym notebooku data dnia = data wydarzenia + przesunięcie:

0: sam dzień wydarzenia.
−1: dzień przed wydarzeniem, −3: trzy dni przed.
+1: dzień po wydarzeniu, +2: dwa dni po.

Notebook sprawdza przesunięcia od −9 do +3, czyli od 9 dni przed do 3 dni po.

In [9]:
OFFSETY = list(range(-9, 4))
n_as = pan.shape[1]
Rv = R.values
indeks = {d: i for i, d in enumerate(kal.index)}
wyniki, okna = [], []
for nazwa, per_rok in WYDARZENIA.items():
    ratio = np.full((len(LATA), len(OFFSETY), n_as), np.nan)
    for ir, rok in enumerate(LATA):
        for io, o in enumerate(OFFSETY):
            idx = [indeks[t + pd.Timedelta(days=o)] for t in per_rok[rok] if (t + pd.Timedelta(days=o)) in indeks]
            if idx:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    ratio[ir, io] = np.nanmean(Rv[idx], axis=0)
    ln = np.log(np.clip(ratio, 0.05, None))
    potw = (np.abs(ln[0]) > prog) & (np.abs(ln[1]) > prog) & (np.sign(ln[0]) == np.sign(ln[1])) & ~np.isnan(ratio).any(axis=0)
    for ia, asid in enumerate(pan.columns):
        zakres = [(o, io) for io, o in enumerate(OFFSETY) if potw[io, ia]]
        for o, io in zakres:
            wyniki.append({"AsId": asid, "wydarzenie": nazwa, "offset": o, "mnoznik": float(np.nanmean(ratio[:, io, ia])),
                           "mnoznik_2023": float(ratio[0, io, ia]), "mnoznik_2024": float(ratio[1, io, ia])})
        if zakres:
            potw_o = {o for o, _ in zakres}
            przed = 0
            while -(przed + 1) in potw_o: przed += 1
            po = 0
            while (po + 1) in potw_o: po += 1
            okna.append({"AsId": asid, "wydarzenie": nazwa, "okno_przed_dni": przed, "okno_po_dni": po,
                         "efekt_w_dniu": 0 in potw_o, "potwierdzone_dni": len(zakres)})
wyn = pd.DataFrame(wyniki)
ok = pd.DataFrame(okna)
print(f"Potwierdzonych par (asortyment, wydarzenie, przesunięcie): {len(wyn)} | par (asortyment, wydarzenie): {len(ok)}")
sk = wyn[wyn["AsId"] == "SKLEP"]
print("\nSklep, potwierdzone wydarzenia (mnożnik wg przesunięcia w dniach):")
for nazwa, g in sk.groupby("wydarzenie"):
    print(f"  {nazwa:22s}", ", ".join(f"{int(o):+d}: ×{m:.2f}" for o, m in zip(g["offset"], g["mnoznik"])))

Potwierdzonych par (asortyment, wydarzenie, przesunięcie): 943 | par (asortyment, wydarzenie): 652

Sklep, potwierdzone wydarzenia (mnożnik wg przesunięcia w dniach):
  Boże Ciało             -1: ×1.49
  Boże Narodzenie        -7: ×1.15, -5: ×1.14, -4: ×1.18, -2: ×1.38, +2: ×0.74, +3: ×0.82
  Drugi dzień BN         -8: ×1.15, -6: ×1.14, -5: ×1.18, -3: ×1.38, +1: ×0.74, +2: ×0.82
  Dzień Kobiet           -6: ×0.90
  Długi weekend          -1: ×1.25
  Halloween              -1: ×1.14, +0: ×1.37
  Konstytucja 3 maja     -1: ×1.21
  Niedziela handlowa     -7: ×0.21, +0: ×0.22
  Nowy Rok               +2: ×0.81, +3: ×0.83
  Sylwester              -8: ×1.38, -4: ×0.74, -3: ×0.82, -1: ×1.20
  Trzech Króli           -3: ×0.81, -2: ×0.83
  Wielkanoc              -7: ×0.25, -6: ×1.11, -5: ×1.26, -4: ×1.22, -3: ×1.41, -2: ×1.56, +2: ×0.84
  Wigilia                -6: ×1.15, -4: ×1.14, -3: ×1.18, -1: ×1.38, +3: ×0.74
  Wniebowzięcie          -1: ×1.32
  Wszystkich Świętych    -2: ×1.14, -1: ×1.37


**Mnożnik** to stosunek rzeczywistej sprzedaży w danym dniu do sprzedaży „normalnej“ dla tego dnia. Mówi, o ile wydarzenie podbiło lub obniżyło sprzedaż.

- 1,00: sprzedaż taka jak zwykle.
- 1,50: o 50% więcej niż zwykle.
- 0,00: brak sprzedaży (np. sklep zamknięty w święto).

Skąd bierze się „normalna“ sprzedaż. Dla każdego dnia notebook wylicza normę: medianę sprzedaży z tego samego dnia tygodnia (np. wszystkich czwartków) w oknie ±6 tygodni, liczoną tylko z dni czystych (bez świąt, ich okolic, okna Wielkanocy i promocji). Dzięki temu czwartek nie jest porównywany z sobotą, a dzień świąteczny nie jest porównywany z innym dniem świątecznym

## Zapis
#### Objaśnienie
**Po co.** Zapisujemy wyłącznie potwierdzone efekty. Wydarzenie, którego brak w pliku, traktujemy w prognozie jak zwykły dzień (mnożnik 1).
**Tabele.** `wynik_06_wydarzenia.csv` ma jeden wiersz na potwierdzone przesunięcie (z mnożnikiem z obu lat), a `wynik_06_okna.csv` podsumowuje długość okna na asortyment i wydarzenie. Wielkanoc z tej tabeli jest podstawą dla wariantu `baseline_1`.

In [10]:
ew.zapisz_wynik(wyn, "wynik_06_wydarzenia.csv", index=False)
ew.zapisz_wynik(ok, "wynik_06_okna.csv", index=False)
print("\nWielkanoc, sklep (okno):")
print(ok[(ok["AsId"] == "SKLEP") & (ok["wydarzenie"] == "Wielkanoc")].to_string(index=False))

Zapisano: dane/eda/wynik_06_wydarzenia.csv (943 wierszy)
Zapisano: dane/eda/wynik_06_okna.csv (652 wierszy)

Wielkanoc, sklep (okno):
 AsId wydarzenie  okno_przed_dni  okno_po_dni  efekt_w_dniu  potwierdzone_dni
SKLEP  Wielkanoc               0            0         False                 7
